### Tokenization and text pre processing for fine-tuning for NLP models

* Tokenization
    * Converts raw text into numerical representations 
    * Types 
        * WordPiece Tokenization : Used in BERT
        * Byte-Pair Encoding (BPE) : Used in GPT and RoBERETa
* Text Pre-procesing
    * Cleaning
        * Remove unnecessary characters ( eg: URLs, special symbols)
        * Normalization
            * Convert text to lowercase
            * Removes stopwords if necessary
        * Tokenization
            * Break Text ito compatible with the pre-trained model
---
### Adapting Pre-Trained models for NLP tasks
* Common Tasks
    * Text Classification 
        * Categorize text into predefined labels (eg: positive/negative sentiment)
    * Setiment Analysis
        * Determine the sentiment polarity of text (eg: Positive , neutral , negative)
    * Summarization
        * Generate concise summarise from lengthy texts
* Steps
    * Load pre-trained model
    * Add a task-specific head (eg: classification layer)
    * Fine Tune the model on task specific data

In [2]:
!uv add transformers datasets

Resolved 125 packages in 16ms
Audited 122 packages in 34ms


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset


In [5]:
dataset = load_dataset("stanfordnlp/imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 169932.89 examples/s]


In [6]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length" , max_length=128,)

In [13]:
tokenize_dataset = dataset.map(tokenize_function, batched=True)

In [14]:
tokenize_dataset = tokenize_dataset.remove_columns(["text"])
tokenize_dataset = tokenize_dataset.rename_column("label","labels")
tokenize_dataset.set_format("torch")

In [15]:

train_dataset = tokenize_dataset["train"]
test_dataset  = tokenize_dataset["test"]


In [10]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=20)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1480.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [17]:
training_args = TrainingArguments(
    output_dir="./",
    eval_strategy="epoch",
    per_device_eval_batch_size=8,
    per_device_train_batch_size= 8, 
    num_train_epochs=1,
    weight_decay= 0.01,
    save_total_limit=2,
)
trainer = Trainer(
    model,
    args = training_args,
    train_dataset= train_dataset,
    eval_dataset = test_dataset,
    processing_class = tokenizer
)

In [18]:
trainer.train()

/home/achiket/Documents/udemy/full-stack-ai-engineer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.313585,0.296337


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]
/home/achiket/Documents/udemy/full-stack-ai-engineer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3125, training_loss=0.38298951416015625, metrics={'train_runtime': 19083.7991, 'train_samples_per_second': 1.31, 'train_steps_per_second': 0.164, 'total_flos': 1644709862400000.0, 'train_loss': 0.38298951416015625, 'epoch': 1.0})

In [19]:
results = trainer.evaluate()
print("Evaluation results", results)

/home/achiket/Documents/udemy/full-stack-ai-engineer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
0.313585,0.296337,1


Evaluation results {'eval_loss': 0.2963371276855469}


In [22]:
import torch

sample_text = dataset["test"][0]["text"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits

predicted_label = int(torch.argmax(logits, dim=-1))

print("Sample text:", sample_text)
print("Predicted label:", predicted_label)
print("Logits:", logits.tolist())
print(dataset["train"].features["label"])
print("Label names:", dataset["train"].features["label"].names)
print("Unique labels in train:", sorted(set(dataset["train"]["label"])))

Sample text: I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi TV. It's not. It's clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It's really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it's rubbish as they 

In [24]:
from huggingface_hub import login, HfApi, create_repo,push_to_hub_fastai

login("<token>")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [25]:
from huggingface_hub import login, HfApi
import os
from shutil import rmtree

# Login 
login("<token>")

# Repo to create/upload to (format: username/repo_name)
repo_id = "Achiket/bert-base-uncased-imdb-finetuned"

api = HfApi()
api.create_repo(repo_id=repo_id, exist_ok=True, private=False)

# Save only the model + tokenizer into a clean folder
model_dir = "hf_model"
if os.path.isdir(model_dir):
    rmtree(model_dir)

model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

# Upload the folder contents to the HF repo

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


NameError: name 'token' is not defined

In [30]:
api.upload_folder(folder_path=model_dir,   repo_id=repo_id, token="<token>")

print(f"Model uploaded to https://huggingface.co/{repo_id}")

Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 2.58MB/s  
New Data Upload: 100%|██████████|  438MB /  438MB, 2.58MB/s  


Model uploaded to https://huggingface.co/Achiket/bert-base-uncased-imdb-finetuned


In [32]:
# Create and upload a model card (README.md) to the HF repo using existing variables: api, repo_id, model_dir, training_args, results, tokenizer, sample_text, predicted_label, logits, model
model_card = f"""# {repo_id.split('/')[-1]}

## Model
- Base: bert-base-uncased (HuggingFace Transformers)
- Fine-tuned for: IMDB sentiment dataset (stanfordnlp/imdb)
- Model class: {model.__class__.__name__}
- Output dim (classifier): {model.classifier.out_features}

## Dataset
- Source: stanfordnlp/imdb (train: {len(dataset['train'])}, test: {len(dataset['test'])}, unsupervised: {len(dataset['unsupervised'])})
- Features: {dataset['train'].column_names}
- Note: IMDB is a binary sentiment dataset (labels: {dataset['train'].features['label'].names if hasattr(dataset['train'].features['label'], 'names') else 'N/A'})

## Tokenization / Preprocessing
- Tokenizer: {tokenizer.__class__.__name__} ({tokenizer.name_or_path})
- Max length used: 128, truncation=True, padding=max_length

## Fine-tuning details
- TrainingArguments (selected):
    - per_device_train_batch_size: {training_args.per_device_train_batch_size}
    - per_device_eval_batch_size: {training_args.per_device_eval_batch_size}
    - num_train_epochs: {training_args.num_train_epochs}
    - learning_rate: {training_args.learning_rate}
    - weight_decay: {training_args.weight_decay}
    - eval_strategy: {training_args.eval_strategy}
    - save_total_limit: {training_args.save_total_limit}

- Important observation: model instantiated with num_labels={model.classifier.out_features}. The IMDB dataset is binary; ensure num_labels matches dataset labels (2) to avoid label-mismatch issues.

## Evaluation
- Eval results (trainer.evaluate): {results}
- Example single inference (from this notebook):
    - sample_text (truncated): {sample_text[:300].replace('\\n',' ')}...
    - predicted_label: {predicted_label}
    - logits: {logits.tolist()}

"""

In [33]:
# write model card to model_dir and upload to HF repo
readme_path = os.path.join(model_dir, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card)

api.upload_file(path_or_fileobj=readme_path, path_in_repo="README.md", repo_id=repo_id)

print(f"Model card uploaded: https://huggingface.co/{repo_id}/blob/main/README.md")

/home/achiket/Documents/udemy/full-stack-ai-engineer/.venv/lib/python3.13/site-packages/huggingface_hub/hf_api.py:11186: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Model card uploaded: https://huggingface.co/Achiket/bert-base-uncased-imdb-finetuned/blob/main/README.md
